In [ ]:
import os
import json
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from bs4 import BeautifulSoup
import requests

In [ ]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
def fetch_website_content(url):
  response = requests.get(url, headers = headers)
  soup = BeautifulSoup(response.content, "html.parser")
  title = soup.title.string if soup.title else "No title found"
  if soup.body:
    for irrelevant in soup.body(["script","style","img","input"]):
      irrelevant.decompose()
    text = soup.body.get_text(separator="\n",strip=True)
  else:
    text = ""
  return (title + "\n\n" + text)[:2_000]

In [ ]:

def fetch_website_links(url):
  response = requests.get(url,headers=headers)
  soup = BeautifulSoup(response.content,"html.parser")
  links= [link.get("href") for link in soup.find_all("a")]
  return [link for link in links if link]

In [ ]:
# Initialize & constants
# The API key is read from Colab Secrets (or an environment variable) - never hardcode it.
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

api_key = GOOGLE_API_KEY

MODEL = 'gemini-3.5-flash-lite'

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url = GEMINI_BASE_URL,api_key = GOOGLE_API_KEY)

In [ ]:
links = fetch_website_links("https://www.geeksforgeeks.org/")
links

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About Page, or a company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a
    brochure about the company, respond with the full https URL
    in JSON format.
    Do not include Terms of Service, Privacy, email links.
    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.geeksforgeeks.org/"))


In [ ]:
def select_relevant_links(url):
  response = openai.chat.completions.create(
      model = MODEL,
      messages = [
          {"role": "system", "content": link_system_prompt},
          {"role": "user", "content": get_links_user_prompt(url)}
      ],
      response_format={"type": "json_object"}
  )
  result=response.choices[0].message.content
  links = json.loads(result)
  return links

In [ ]:
select_relevant_links("https://www.geeksforgeeks.org/")

{'links': [{'type': 'about page',
   'url': 'https://www.geeksforgeeks.org/about/'},
  {'type': 'contact page',
   'url': 'https://www.geeksforgeeks.org/about/contact-us/'},
  {'type': 'courses page', 'url': 'https://www.geeksforgeeks.org/courses'},
  {'type': 'corporate solutions',
   'url': 'https://www.geeksforgeeks.org/gfg-corporate-solution/'},
  {'type': 'campus training',
   'url': 'https://www.geeksforgeeks.org/campus-training-program/'}]}

In [ ]:

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_content(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_content(link["url"])

    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://www.geeksforgeeks.org/"))


In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("GFG", "https://www.geeksforgeeks.org/")


In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("GFG", "https://www.geeksforgeeks.org/")

# Welcome to GeeksforGeeks (GFG)
Your All-in-One Learning and Career Portal

## Empowering Millions Worldwide
Founded in 2008 by visionary computer scientist Mr. Sandeep Jain, GeeksforGeeks (GfG) is a comprehensive educational portal dedicated to transforming curiosity into expertise. Serving over 50 million registered users globally and millions of daily visitors, GfG is a trusted name in education, offering a vibrant community where students and professionals can learn, grow, and stay ahead in the ever-evolving world of technology.

## What We Offer
We bridge the gap between academic learning and industry demands through a vast collection of tutorials, concept explainers, coding challenges, practice problems, and structured courses spanning:
*   Data Structures and Algorithms (DSA) & System Design
*   AI, Machine Learning, Generative AI & Agentic AI
*   Full Stack Web Development (MERN, Java Backend)
*   DevOps & Cloud Computing (AWS, Docker, Kubernetes)
*   Programming Languages (C, C++, Java, Python, JavaScript)
*   Computer Science Core Subjects, Databases, and Software Tools
*   Exam Preparation Resources (GATE, JEE, NEET, and more)

## Our Customers & Community
Our platform caters to a diverse audience, including school students, university scholars, and working professionals looking to upskill or switch domains. GfG is globally renowned for our top-notch interview preparation services. Through curated content, company-wise interview experiences, practice DSA, and coding contests like Job-A-Thon and the GfG Coding Contest, we help thousands of learners successfully land roles at top technology companies. We also partner with institutions and enterprises through our GFG Corporate Solutions and Campus Training Programs.

## Company Culture & Careers
At GeeksforGeeks, our culture is rooted in collaboration, innovation, and impact. Our content and courses are crafted and curated by top mentors from renowned institutions and organizations to ensure the highest standards of quality and relevance. 

We are always looking for passionate individuals to join our mission of empowering learners worldwide. Whether you are an educator, content creator, developer, or mentor, GfG offers exciting career and internship opportunities. Explore our hiring channels for roles in content creation, technical instruction, software engineering, and more.

## Get in Touch
*   **Corporate & Communications Address:** A-143, 6th Floor, Sovereign Corporate Tower, Sector- 136, Noida, Uttar Pradesh (201305)
*   **Registered Address:** K 061, Tower K, Gulshan Vivante Apartment, Sector 137, Noida, Gautam Buddh Nagar, Uttar Pradesh, 201305
*   **Website:** Visit us to explore courses, tutorials, and career opportunities today!